<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://www.sap.com/dam/application/shared/logos/sap-logo-svg.svg" alt="SAP" width="100">
</div>

# Predictive JIT Supply Chain Risk with SAP-RPT-1
## Part 1: From Dark Data to Delay Prediction

### The Use Case: Intelligent JIT Risk Prediction

**BestRun Technologies** operates a Just-In-Time (JIT) manufacturing model for smart IoT security devices. Their production schedules are hyper-optimized—a single delayed shipment of a critical component creates a "line-down" situation costing **$15,000+ per hour** in idle labor and missed delivery penalties.

**The Problem:** Current supply chain management is reactive. Managers only realize a Purchase Order (PO) is delayed when the truck doesn't arrive. While S/4HANA contains years of historical supplier performance data, it remains "dark data"—underutilized for forward-looking insights.

**The Solution:** In this workshop, you will use SAP's **sap-rpt-1** (Report Transformer) model to predict which POs are at risk of delay *before* they happen, enabling proactive mitigation.

---

### Workshop Roadmap

| Section | Time | Outcome |
|---------|------|--------|
| **Setup** (Steps 1.1-1.3) | 10 min | SDK + config + AI Core connectivity |
| **Data Exploration** (Steps 1.4-1.6) | 15 min | Understand historical PO patterns and risk distribution |
| **Prediction** (Steps 1.7-1.9) | 20 min | Run SAP-RPT-1 inference on new POs |
| **Business Policy** (Steps 1.10-1.12) | 15 min | Apply risk tiers and generate mitigation proposals |
| **Checkpoint & Exercise** | 10 min | Explore different scenarios |

---

### Learning Objectives

By the end of Part 1, you will be able to:
- Prepare structured procurement data for tabular AI prediction
- Invoke SAP-RPT-1 via SAP AI Core for delay risk scoring
- Apply business policies to translate predictions into actionable risk tiers
- Generate mitigation proposals using alternative supplier data

## Business Context: Why This Matters

### The Cost of Being Reactive

| Scenario | Impact |
|----------|--------|
| Critical sensor delayed 2 days | $240,000 lost production |
| Control chip arrives 1 week late | Missed customer SLA, $50,000 penalty |
| Proactive supplier switch | $5,000 premium, but $0 downtime |

### The Data Landscape

BestRun's S/4HANA system contains:
- **600+ historical POs** with actual delivery outcomes
- **3 key suppliers** with varying reliability profiles
- **6 material types** ranging from critical sensors to commodity connectors
- **Seasonal patterns** (Q4 logistics congestion, monsoon impacts in Asia)

### SAP BTP Services Used

| Service | Role | Why SAP-Native? |
|---------|------|----------------|
| **SAP AI Core** | Model hosting and inference runtime | Enterprise auth, resource isolation, audit trail |
| **SAP-RPT-1** | Tabular prediction model | Purpose-built for structured business data |
| **Gen AI Hub SDK** | Pro-code orchestration | Consistent API, version-pinned, SAP support |

### Architecture Decision Lens: Why `sap-rpt-1` in This Scenario

Before implementation, validate the decision pattern as an architecture checkpoint:

| Question | If yes | If no |
|---------|--------|-------|
| Is the data primarily structured, tabular business data from ERP? | `sap-rpt-1` is a strong fit | Consider LLM or unstructured retrieval patterns |
| Is the target output a numeric risk signal or forecast? | Use predictive scoring first | Consider descriptive or generative patterns |
| Is the process mostly known and policy-driven? | Start deterministic (Part 1 pattern) | Consider adaptive orchestration (Part 2) |
| Is full autonomy required? | Keep human approval for sourcing changes | Keep advisory-only recommendation flow |

**Architect guidance:** start with the simplest architecture that can prove business value, then add agentic complexity only where it measurably improves decisions.


---

## Part 1: Setup & Configuration

### Step 1.1 - Install Required Packages

We use SAP's `generative-ai-hub-sdk` for AI Core integration and standard data science libraries.

**Why these packages:**
- `generative-ai-hub-sdk`: SAP-supported SDK for AI Core and Gen AI Hub
- `pandas`: Data manipulation for tabular PO data
- `python-dotenv`: Secure credential management

**Restart the kernel** after installation if prompted.

In [ ]:
%pip install generative-ai-hub-sdk==4.12.4 --quiet
%pip install pandas python-dotenv requests --quiet

print("Packages installed successfully.")

### Step 1.1b - Prerequisites Check

Run this cell to verify your environment is ready before proceeding.

In [ ]:
import sys
import socket

def check_prerequisites():
    """Validate environment before workshop starts."""
    checks_passed = 0
    checks_total = 3
    
    # Check 1: Python version
    py_version = sys.version_info
    if py_version >= (3, 9):
        print(f"[PASS] Python version: {py_version.major}.{py_version.minor}.{py_version.micro}")
        checks_passed += 1
    else:
        print(f"[FAIL] Python version {py_version.major}.{py_version.minor} - requires 3.9+")
    
    # Check 2: Required packages importable
    try:
        import pandas as pd
        import requests
        from dotenv import load_dotenv
        print("[PASS] Required packages are importable")
        checks_passed += 1
    except ImportError as e:
        print(f"[FAIL] Missing package: {e.name}. Re-run the install cell above.")
    
    # Check 3: Data files exist
    import os
    data_files = ["data/historical_po_data.csv", "data/new_po_prediction.csv", "data/alt_supplier_table.csv"]
    missing = [f for f in data_files if not os.path.exists(f)]
    if not missing:
        print("[PASS] All data files found")
        checks_passed += 1
    else:
        print(f"[FAIL] Missing data files: {missing}")
    
    print("-" * 50)
    if checks_passed == checks_total:
        print(f"All {checks_total} checks passed. Ready to proceed!")
    else:
        print(f"{checks_passed}/{checks_total} checks passed. Review issues above.")

check_prerequisites()

### Step 1.2 - Imports and Global Constants

This cell imports all required libraries and defines constants used throughout the notebook.

In [ ]:
import json
import os
import time
from typing import Any, Dict, List, Tuple

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display, HTML

# --- Global Constants ---
MODEL_NAME = "sap-rpt-1-small"
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 60
TOKEN_REFRESH_BUFFER_SECONDS = 60

# --- Risk Tier Thresholds (Business Policy) ---
RISK_THRESHOLD_AMBER = 1.0  # Days
RISK_THRESHOLD_RED = 3.0   # Days

# --- Cost Parameters (for business case calculation) ---
LINE_DOWN_COST_PER_HOUR = 15000  # USD
TYPICAL_DISRUPTION_HOURS = 8

print("Imports and constants loaded.")

### Step 1.3 - Load `.env` Configuration

We separate credentials from code using a `.env` file, a best practice for enterprise applications.

**Expected environment variables:**
- `AICORE_AUTH_URL` - OAuth token endpoint
- `AICORE_CLIENT_ID` - Service key client ID
- `AICORE_CLIENT_SECRET` - Service key client secret
- `AICORE_BASE_URL` - AI Core API base URL
- `AICORE_RESOURCE_GROUP` - Your resource group name
- `RPT1_DEPLOYMENT_URL` - SAP-RPT-1 model deployment base endpoint

> **Note:** Your instructor will provide the `.env` file or guide you through creating one.
>
> **Endpoint format note:** Set `RPT1_DEPLOYMENT_URL` to the deployment base URL (for example `.../v2/inference/deployments/<deployment_id>`). The notebook appends `/predict` automatically when calling SAP-RPT-1.

In [ ]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path=dotenv_path, override=True)

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
RPT1_DEPLOYMENT_URL = os.getenv("RPT1_DEPLOYMENT_URL")

required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
}

missing = [k for k, v in required.items() if not v]
if missing:
    print(f"[WARNING] Missing environment variables: {missing}")
    print("The notebook will use MOCK mode for predictions.")
    USE_MOCK_MODE = True
else:
    USE_MOCK_MODE = False
    print(f"Loaded .env from: {dotenv_path}")
    print(f"Resource group: {AICORE_RESOURCE_GROUP}")

if not RPT1_DEPLOYMENT_URL:
    print("[INFO] RPT1_DEPLOYMENT_URL not set - using mock prediction mode")
    USE_MOCK_MODE = True
else:
    print(f"RPT-1 Deployment: ...{RPT1_DEPLOYMENT_URL[-20:]}")

print(f"\nMock mode: {USE_MOCK_MODE}")

### Troubleshooting Common Issues

| Error | Likely Cause | Fix |
|-------|--------------|-----|
| `Missing environment variables` | `.env` file not found or incomplete | Ensure `.env` exists in project root with all required keys |
| `401 Unauthorized` | Invalid client credentials | Verify `AICORE_CLIENT_ID` and `AICORE_CLIENT_SECRET` |
| `403 Forbidden` | Wrong resource group | Verify `AICORE_RESOURCE_GROUP` matches your AI Core setup |
| `Connection timeout` | Network/VPN issue | Check VPN connection to SAP network |
| `Model not found` | Deployment not active | Check AI Launchpad for deployment status |

---

## Part 2: Data Exploration

Before we predict, let's understand the data. This mirrors what a supply chain analyst would do when preparing data from S/4HANA.

### Step 1.4 - Load the Datasets

We have three datasets:
1. **Historical PO Data** - Past purchase orders with known delivery outcomes
2. **New PO for Prediction** - A newly created PO we need to assess
3. **Alternative Supplier Table** - Backup suppliers for each material

In [ ]:
# Load datasets
historical_df = pd.read_csv("data/historical_po_data.csv")
prediction_df = pd.read_csv("data/new_po_prediction.csv")
alt_supplier_df = pd.read_csv("data/alt_supplier_table.csv")

print(f"Historical POs: {len(historical_df):,} records")
print(f"POs to predict: {len(prediction_df):,} records")
print(f"Alternative suppliers: {len(alt_supplier_df):,} records")
print("\n" + "="*60)
print("Sample of historical data:")
historical_df.head()

### Step 1.5 - Understand the Feature Set

The SAP-RPT-1 model uses these features to predict delay risk:

| Feature | Source (S/4HANA) | Description |
|---------|------------------|-------------|
| `Vendor_ID` | LFA1 | Supplier identifier |
| `Vendor_Country` | LFA1 | Supplier location (logistics risk) |
| `Vendor_OTIF_Percent` | Derived | On-Time-In-Full historical rate |
| `Vendor_Avg_Past_Delay` | Derived | Average historical delay (days) |
| `Material_ID` | MARA | Material number |
| `Material_Group` | MARA | Material category |
| `Criticality_Flag` | Custom | Is this a line-stopping component? |
| `Order_Quantity` | EKPO | Units ordered |
| `Net_Price` | EKPO | Unit price (USD) |
| `Planned_Lead_Time_Days` | EKPO/INFO | Expected delivery time |
| `Order_Month` | EKKO | Seasonality indicator |
| `Incoterms` | EKKO | Delivery responsibility terms |

In [ ]:
print("Dataset Schema:")
print("-" * 40)
for col in historical_df.columns:
    dtype = historical_df[col].dtype
    sample = historical_df[col].iloc[0]
    print(f"{col:30} | {str(dtype):10} | Example: {sample}")

### Step 1.6 - Analyze Historical Risk Distribution

Understanding the historical delay patterns helps us set realistic expectations for predictions.

**Risk Tier Policy:**
- **Green** (< 1 day delay): Standard monitoring
- **Amber** (1-3 days delay): Increased attention, consider backup
- **Red** (> 3 days delay): Immediate action required

In [ ]:
def derive_risk_tier(delay_days: float) -> str:
    """Classify delay into business risk tiers."""
    if delay_days < RISK_THRESHOLD_AMBER:
        return "Green"
    elif delay_days <= RISK_THRESHOLD_RED:
        return "Amber"
    return "Red"

# Apply to historical data
historical_df["Risk_Tier"] = historical_df["Actual_Delay_Days"].apply(derive_risk_tier)

# Display distribution
risk_counts = historical_df["Risk_Tier"].value_counts()
risk_pcts = (risk_counts / len(historical_df) * 100).round(1)

print("Historical Risk Distribution:")
print("=" * 40)
for tier in ["Green", "Amber", "Red"]:
    count = risk_counts.get(tier, 0)
    pct = risk_pcts.get(tier, 0)
    bar = "|" * int(pct / 2)
    print(f"{tier:8} | {count:4} POs ({pct:5.1f}%) {bar}")

print("\n" + "=" * 40)
print(f"Average delay: {historical_df['Actual_Delay_Days'].mean():.2f} days")
print(f"Max delay: {historical_df['Actual_Delay_Days'].max():.1f} days")

### Step 1.6b - Vendor Performance Analysis

Different suppliers have different reliability profiles. This is key input for the prediction model.

In [ ]:
vendor_stats = historical_df.groupby("Vendor_ID").agg({
    "Actual_Delay_Days": ["mean", "std", "max"],
    "PO_ID": "count",
    "Vendor_Country": "first",
    "Vendor_OTIF_Percent": "first"
}).round(2)

vendor_stats.columns = ["Avg_Delay", "Std_Delay", "Max_Delay", "PO_Count", "Country", "OTIF_%"]
vendor_stats = vendor_stats.sort_values("Avg_Delay")

print("Vendor Performance Summary:")
print("=" * 70)
display(vendor_stats)

print("\nKey Insight:")
best = vendor_stats.index[0]
worst = vendor_stats.index[-1]
print(f"  - {best} (USA) has the best performance: {vendor_stats.loc[best, 'Avg_Delay']:.1f} day avg delay")
print(f"  - {worst} (Vietnam) has highest risk: {vendor_stats.loc[worst, 'Avg_Delay']:.1f} day avg delay")

---

## Part 3: Running SAP-RPT-1 Prediction

Now we'll use the SAP-RPT-1 model to predict delay risk for new purchase orders.

### Step 1.7 - SAP AI Core Client

This client handles authentication and API calls to SAP AI Core.

In [ ]:
class AICoreClient:
    """
    Client for SAP AI Core API.
    
    Handles OAuth token management and inference requests.
    """
    
    def __init__(self, auth_url: str, client_id: str, client_secret: str,
                 base_url: str, resource_group: str):
        self._auth_url = auth_url
        self._client_id = client_id
        self._client_secret = client_secret
        self._base_url = base_url
        self._resource_group = resource_group
        self._access_token: str | None = None
        self._token_expires_at: float = 0.0
    
    def _get_token(self) -> str:
        """Get a valid access token, refreshing if necessary."""
        now = time.time()
        if self._access_token and now < (self._token_expires_at - TOKEN_REFRESH_BUFFER_SECONDS):
            return self._access_token
        
        response = requests.post(
            f"{self._auth_url}/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            timeout=TOKEN_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()
        
        self._access_token = payload["access_token"]
        self._token_expires_at = now + int(payload.get("expires_in", 600))
        return self._access_token
    
    def predict(self, deployment_url: str, payload: dict) -> dict:
        """
        Run inference on SAP AI Core deployment.
        
        Args:
            deployment_url: Full URL to the model deployment endpoint
            payload: Request payload for the model
            
        Returns:
            Model response as dictionary
        """
        token = self._get_token()
        
        response = requests.post(
            deployment_url,
            json=payload,
            headers={
                "Authorization": f"Bearer {token}",
                "AI-Resource-Group": self._resource_group,
                "Content-Type": "application/json"
            },
            timeout=HTTP_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        return response.json()


# Initialize client if credentials are available
if not USE_MOCK_MODE:
    aicore_client = AICoreClient(
        auth_url=AICORE_AUTH_URL,
        client_id=AICORE_CLIENT_ID,
        client_secret=AICORE_CLIENT_SECRET,
        base_url=AICORE_BASE_URL,
        resource_group=AICORE_RESOURCE_GROUP,
    )
    print("AI Core client initialized.")
else:
    aicore_client = None
    print("Running in MOCK mode - no AI Core client needed.")

### Step 1.8 - Define Prediction Functions

We define both the real SAP-RPT-1 prediction and a mock fallback for demonstration purposes.

**How SAP-RPT-1 Works:**
- Takes historical context (past POs with outcomes) + new PO features
- Uses transformer architecture to learn patterns from tabular data
- Returns predicted delay in days

In [ ]:
def mock_rpt1_predict(row: pd.Series) -> float:
    """
    Mock prediction function for demonstration.

    This simulates the behavior of SAP-RPT-1 based on known patterns
    in the training data. In production, this would be replaced by
    the actual model inference.

    Patterns encoded:
    - VENDOR_A (USA, 96% OTIF): Low delay baseline
    - VENDOR_B (Malaysia, 88% OTIF): Medium delay, worse in Q4
    - VENDOR_C (Vietnam, 82% OTIF): Higher delay, worse with large orders
    - Critical materials and FOB terms add risk
    """
    vendor = row["Vendor_ID"]
    month = int(row["Order_Month"])
    qty = int(row["Order_Quantity"])
    material_group = row["Material_Group"]
    incoterms = row["Incoterms"]
    criticality = row["Criticality_Flag"]

    # Base delay by vendor
    delay = 0.0
    if vendor == "VENDOR_A":
        delay += 0.5
    elif vendor == "VENDOR_B":
        delay += 1.8
    else:  # VENDOR_C
        delay += 2.6

    # Seasonal effects (Q4 logistics congestion)
    if vendor == "VENDOR_B" and month in [10, 11, 12]:
        delay += 1.5

    # Volume effects (capacity constraints)
    if vendor == "VENDOR_C" and qty > 1500:
        delay += 1.2

    # Material complexity
    if material_group in ["Control Chip", "Optical Module"]:
        delay += 0.3

    # Incoterms impact (who bears transit risk)
    if incoterms == "FOB":
        delay += 0.2
    elif incoterms == "DAP":
        delay -= 0.2

    return round(max(delay, 0), 1)


def predict_delay(row: pd.Series, context_df: pd.DataFrame) -> Tuple[float, str]:
    """
    Predict delivery delay for a purchase order.

    Args:
        row: The new PO to predict
        context_df: Historical POs for context

    Returns:
        Tuple of (predicted_delay_days, prediction_method)
    """
    if USE_MOCK_MODE or aicore_client is None:
        return mock_rpt1_predict(row), "mock"

    try:
        # Prepare payload for SAP-RPT-1
        columns = [
            "Vendor_ID", "Vendor_Country", "Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay",
            "Material_ID", "Material_Group", "Criticality_Flag", "Plant_ID",
            "Order_Quantity", "Net_Price", "Planned_Lead_Time_Days", "Order_Month",
            "Incoterms", "Actual_Delay_Days"
        ]

        # Sample context for efficiency
        context_sample = context_df[columns].sample(n=min(200, len(context_df)), random_state=42)

        # Prepare prediction row
        pred_row = row[columns[:-1]].to_dict()
        pred_row["Actual_Delay_Days"] = "[PREDICT]"

        # SAP-RPT-1 expects rows + prediction_config payload
        rows = pd.concat([context_sample, pd.DataFrame([pred_row])], ignore_index=True).to_dict("records")
        payload = {
            "rows": rows,
            "prediction_config": {
                "target_columns": [
                    {
                        "name": "Actual_Delay_Days",
                        "prediction_placeholder": "[PREDICT]"
                    }
                ]
            }
        }

        # Deployment URL in .env is base path; model endpoint is /predict
        deployment_url = RPT1_DEPLOYMENT_URL.rstrip("/")
        if not deployment_url.endswith("/predict"):
            deployment_url = f"{deployment_url}/predict"

        # Call SAP-RPT-1
        response = aicore_client.predict(
            deployment_url=deployment_url,
            payload=payload
        )

        prediction_value = (
            response["predictions"][0]["Actual_Delay_Days"][0]["prediction"]
        )
        predicted_delay = float(prediction_value)
        return predicted_delay, "sap-rpt-1"

    except requests.HTTPError as e:
        error_detail = ""
        if e.response is not None:
            error_detail = f" | status={e.response.status_code} body={e.response.text[:300]}"
        print(f"[WARNING] SAP-RPT-1 call failed: {e}{error_detail}")
        print("Falling back to mock prediction.")
        return mock_rpt1_predict(row), "mock-fallback"
    except Exception as e:
        print(f"[WARNING] SAP-RPT-1 call failed: {e}")
        print("Falling back to mock prediction.")
        return mock_rpt1_predict(row), "mock-fallback"


print("Prediction functions ready.")
print(f"Mode: {'MOCK' if USE_MOCK_MODE else 'SAP-RPT-1'}")

### Step 1.9 - Configure the Prediction Scenario

Let's set up the PO we want to predict. You can modify the vendor and quantity to explore different scenarios.

**Try these combinations:**
| Vendor | Quantity | Expected Risk |
|--------|----------|---------------|
| VENDOR_A | 1000 | Low (Green) |
| VENDOR_B | 1500 | Medium (Amber) |
| VENDOR_C | 2200 | High (Red) |

In [ ]:
# ============================================================
# WORKSHOP PARAMETERS - Modify these to explore scenarios
# ============================================================

SELECTED_VENDOR = "VENDOR_C"   # Options: VENDOR_A, VENDOR_B, VENDOR_C
SELECTED_QUANTITY = 2200       # Options: 500, 1000, 1500, 2200, 2600

# ============================================================

# Update the prediction row
prediction_df.loc[0, "Vendor_ID"] = SELECTED_VENDOR
prediction_df.loc[0, "Order_Quantity"] = SELECTED_QUANTITY

# Sync vendor profile attributes
vendor_profile = historical_df.groupby("Vendor_ID")[
    ["Vendor_OTIF_Percent", "Vendor_Avg_Past_Delay", "Vendor_Country"]
].first().reset_index()

vendor_attrs = vendor_profile[vendor_profile["Vendor_ID"] == SELECTED_VENDOR].iloc[0]
prediction_df.loc[0, "Vendor_OTIF_Percent"] = vendor_attrs["Vendor_OTIF_Percent"]
prediction_df.loc[0, "Vendor_Avg_Past_Delay"] = vendor_attrs["Vendor_Avg_Past_Delay"]
prediction_df.loc[0, "Vendor_Country"] = vendor_attrs["Vendor_Country"]

print("Prediction Scenario Configured:")
print("=" * 50)
print(f"PO Number:      {prediction_df.loc[0, 'PO_ID']}")
print(f"Vendor:         {SELECTED_VENDOR} ({vendor_attrs['Vendor_Country']})")
print(f"Vendor OTIF:    {vendor_attrs['Vendor_OTIF_Percent']}%")
print(f"Material:       {prediction_df.loc[0, 'Material_ID']} ({prediction_df.loc[0, 'Material_Group']})")
print(f"Critical:       {prediction_df.loc[0, 'Criticality_Flag']}")
print(f"Quantity:       {SELECTED_QUANTITY:,} units")
print(f"Order Month:    {prediction_df.loc[0, 'Order_Month']} (November)")
print(f"Incoterms:      {prediction_df.loc[0, 'Incoterms']}")

### Step 1.9b - Run the Prediction

Now let's predict the delay risk for this PO.

In [ ]:
# Run prediction
predicted_delay, method = predict_delay(prediction_df.iloc[0], historical_df)
risk_tier = derive_risk_tier(predicted_delay)

# Color coding for display
tier_colors = {"Green": "#2e7d32", "Amber": "#f57f17", "Red": "#c62828"}
tier_color = tier_colors.get(risk_tier, "#757575")

print("\n" + "=" * 50)
print("PREDICTION RESULT")
print("=" * 50)
print(f"Predicted Delay:  {predicted_delay} days")
print(f"Risk Tier:        {risk_tier}")
print(f"Prediction Method: {method}")
print(f"Material Critical: {prediction_df.loc[0, 'Criticality_Flag']}")

# Visual display
html_result = f"""
<div style="border:2px solid {tier_color}; border-radius:8px; padding:16px; margin:10px 0; background:#f5f5f5;">
    <h3 style="margin-top:0;">Prediction Result: PO {prediction_df.loc[0, 'PO_ID']}</h3>
    <div style="display:flex; gap:40px;">
        <div>
            <strong>Predicted Delay:</strong><br>
            <span style="font-size:2em; font-weight:bold;">{predicted_delay} days</span>
        </div>
        <div>
            <strong>Risk Tier:</strong><br>
            <span style="font-size:2em; font-weight:bold; color:{tier_color};">{risk_tier}</span>
        </div>
        <div>
            <strong>Material:</strong><br>
            <span>{prediction_df.loc[0, 'Material_Group']}</span><br>
            <span style="color:{'red' if prediction_df.loc[0, 'Criticality_Flag']=='Yes' else 'gray'};">
                {'CRITICAL' if prediction_df.loc[0, 'Criticality_Flag']=='Yes' else 'Standard'}
            </span>
        </div>
    </div>
</div>
"""
display(HTML(html_result))

---

## Part 4: Business Policy & Mitigation

Prediction alone isn't useful—we need to translate it into business action.

### Step 1.10 - Apply Business Policy

BestRun's supply chain policy:

| Condition | Action |
|-----------|--------|
| Green risk + non-critical | Standard monitoring |
| Amber risk OR critical material | Increased attention, notify planner |
| Red risk + critical material | **Immediate mitigation required** |

In [ ]:
def apply_business_policy(risk_tier: str, is_critical: bool) -> Tuple[str, str]:
    """
    Apply BestRun's supply chain risk policy.
    
    Returns:
        Tuple of (action_level, recommendation)
    """
    if risk_tier == "Red" and is_critical:
        return "CRITICAL", "Immediate mitigation required. Initiate alternative sourcing."
    elif risk_tier == "Red":
        return "HIGH", "High risk detected. Review alternative suppliers."
    elif risk_tier == "Amber" or is_critical:
        return "ELEVATED", "Increased monitoring. Prepare contingency plan."
    else:
        return "NORMAL", "Standard supplier monitoring. No action required."


def estimate_mitigation_value(delay_days: float, is_critical: bool) -> int:
    """
    Estimate the business value of proactive mitigation.
    
    Assumes:
    - Line-down cost of $15,000/hour
    - Typical disruption event of 8 hours
    - Critical materials cause full line stoppage
    """
    if not is_critical or delay_days < RISK_THRESHOLD_RED:
        return 0
    
    # Estimate hours of disruption based on delay severity
    disruption_hours = min(delay_days * 4, TYPICAL_DISRUPTION_HOURS * 2)  # Cap at 2x typical
    return int(disruption_hours * LINE_DOWN_COST_PER_HOUR)


# Apply policy
is_critical = prediction_df.loc[0, "Criticality_Flag"] == "Yes"
action_level, recommendation = apply_business_policy(risk_tier, is_critical)
avoided_loss = estimate_mitigation_value(predicted_delay, is_critical)

print("Business Policy Assessment:")
print("=" * 50)
print(f"Action Level:     {action_level}")
print(f"Recommendation:   {recommendation}")
if avoided_loss > 0:
    print(f"\nPotential Avoided Loss: ${avoided_loss:,.0f}")
    print(f"  (Based on {predicted_delay:.1f} days delay x $15,000/hour line-down cost)")

### Step 1.11 - Generate Mitigation Proposal

When high risk is detected, we automatically identify alternative suppliers.

### Decision Rights and Business Trade-Offs

Prediction is advisory. Decision accountability stays with planners and approvers.

| Decision Option | Typical Benefit | Typical Cost/Risk | Default Owner |
|----------------|-----------------|-------------------|---------------|
| Stay with current supplier | No switching effort | Higher delay risk | Planner |
| Expedite current supplier | Faster recovery | Premium freight cost | Planner + Procurement |
| Switch to alternate supplier | Lower delay probability | Price/quality onboarding risk | Procurement Approver |
| Split order across suppliers | Risk diversification | Coordination complexity | Supply Chain Lead |

**Governance rule in this workshop:** model predicts, agent proposes, human approves any sourcing-impacting action.


In [ ]:
def find_alternative_suppliers(material_id: str, current_vendor: str, 
                                alt_df: pd.DataFrame) -> pd.DataFrame:
    """
    Find alternative suppliers for a material.
    
    Filters:
    - Same material
    - Different vendor (not current)
    
    Sorts by:
    - Lead time (ascending)
    - Price (ascending)
    - Stock (descending)
    """
    candidates = alt_df[
        (alt_df["Material_ID"] == material_id) &
        (alt_df["Alt_Vendor"] != current_vendor)
    ].copy()
    
    return candidates.sort_values(
        ["Lead_Time_Days", "Indicative_Unit_Price"],
        ascending=[True, True]
    )


def generate_mitigation_proposal(prediction_row: pd.Series, predicted_delay: float,
                                  risk_tier: str, alt_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Generate a structured mitigation proposal.
    """
    material_id = prediction_row["Material_ID"]
    current_vendor = prediction_row["Vendor_ID"]
    quantity_needed = int(prediction_row["Order_Quantity"])
    is_critical = prediction_row["Criticality_Flag"] == "Yes"
    
    # Find alternatives
    alternatives = find_alternative_suppliers(material_id, current_vendor, alt_df)
    
    if len(alternatives) == 0:
        return {
            "status": "NO_ALTERNATIVES",
            "message": f"No alternative suppliers found for {material_id}"
        }
    
    # Select best alternative
    best = alternatives.iloc[0]
    
    # Calculate cost comparison
    current_unit_price = prediction_row["Net_Price"]
    alt_unit_price = best["Indicative_Unit_Price"]
    price_premium = (alt_unit_price - current_unit_price) * quantity_needed
    
    # Stock availability check
    stock_available = int(best["Current_Available_Stock"])
    can_fulfill = stock_available >= quantity_needed
    
    return {
        "status": "PROPOSAL_READY",
        "risk_summary": {
            "predicted_delay": predicted_delay,
            "risk_tier": risk_tier,
            "is_critical": is_critical,
            "avoided_loss": estimate_mitigation_value(predicted_delay, is_critical)
        },
        "current_order": {
            "vendor": current_vendor,
            "material": material_id,
            "quantity": quantity_needed,
            "unit_price": current_unit_price
        },
        "recommended_alternative": {
            "vendor": best["Alt_Vendor"],
            "country": best["Alt_Vendor_Country"],
            "lead_time_days": int(best["Lead_Time_Days"]),
            "unit_price": alt_unit_price,
            "stock_available": stock_available,
            "can_fulfill": can_fulfill
        },
        "financial_impact": {
            "price_premium": price_premium,
            "avoided_loss": estimate_mitigation_value(predicted_delay, is_critical),
            "net_benefit": estimate_mitigation_value(predicted_delay, is_critical) - max(price_premium, 0)
        },
        "all_alternatives": alternatives.to_dict("records")
    }


# Generate proposal if high risk
if action_level in ["CRITICAL", "HIGH"]:
    proposal = generate_mitigation_proposal(
        prediction_df.iloc[0], predicted_delay, risk_tier, alt_supplier_df
    )
    
    print("\n" + "=" * 60)
    print("MITIGATION PROPOSAL")
    print("=" * 60)
    
    if proposal["status"] == "PROPOSAL_READY":
        rec = proposal["recommended_alternative"]
        curr = proposal["current_order"]
        fin = proposal["financial_impact"]
        
        print(f"\nCurrent Order:")
        print(f"  Vendor: {curr['vendor']}")
        print(f"  Material: {curr['material']}")
        print(f"  Quantity: {curr['quantity']:,} units")
        print(f"  Unit Price: ${curr['unit_price']:.2f}")
        
        print(f"\nRecommended Alternative:")
        print(f"  Vendor: {rec['vendor']} ({rec['country']})")
        print(f"  Lead Time: {rec['lead_time_days']} days")
        print(f"  Unit Price: ${rec['unit_price']:.2f}")
        print(f"  Stock Available: {rec['stock_available']:,} units")
        print(f"  Can Fulfill Order: {'Yes' if rec['can_fulfill'] else 'PARTIAL'}")
        
        print(f"\nFinancial Analysis:")
        print(f"  Price Premium: ${fin['price_premium']:,.0f}")
        print(f"  Avoided Loss: ${fin['avoided_loss']:,.0f}")
        print(f"  Net Benefit: ${fin['net_benefit']:,.0f}")
        
        print(f"\nRECOMMENDED ACTION:")
        print(f"  Initiate purchase requisition with {rec['vendor']}")
else:
    proposal = None
    print("\nNo mitigation proposal needed - risk level is acceptable.")

### Step 1.12 - Display Complete Risk Assessment Dashboard

In [ ]:
def display_risk_dashboard(po_row: pd.Series, predicted_delay: float, risk_tier: str,
                           action_level: str, proposal: Dict | None):
    """Display a formatted risk assessment dashboard."""
    
    tier_colors = {"Green": "#2e7d32", "Amber": "#f57f17", "Red": "#c62828"}
    action_colors = {"NORMAL": "#2e7d32", "ELEVATED": "#f57f17", 
                     "HIGH": "#e65100", "CRITICAL": "#c62828"}
    
    # Build alternatives table if available
    alt_table = ""
    if proposal and proposal.get("status") == "PROPOSAL_READY":
        rec = proposal["recommended_alternative"]
        fin = proposal["financial_impact"]
        
        alt_table = f"""
        <div style="margin-top:20px; padding:15px; background:#fff3e0; border-radius:6px;">
            <h4 style="margin-top:0; color:#e65100;">Recommended Mitigation</h4>
            <table style="width:100%;">
                <tr><td><strong>Alternative Vendor:</strong></td><td>{rec['vendor']} ({rec['country']})</td></tr>
                <tr><td><strong>Lead Time:</strong></td><td>{rec['lead_time_days']} days</td></tr>
                <tr><td><strong>Unit Price:</strong></td><td>${rec['unit_price']:.2f}</td></tr>
                <tr><td><strong>Stock Available:</strong></td><td>{rec['stock_available']:,} units</td></tr>
                <tr><td><strong>Price Premium:</strong></td><td>${fin['price_premium']:,.0f}</td></tr>
                <tr><td><strong>Avoided Loss:</strong></td><td style="color:#2e7d32; font-weight:bold;">${fin['avoided_loss']:,.0f}</td></tr>
                <tr><td><strong>Net Benefit:</strong></td><td style="font-weight:bold;">${fin['net_benefit']:,.0f}</td></tr>
            </table>
        </div>
        """
    
    dashboard_html = f"""
    <div style="font-family:system-ui,-apple-system,sans-serif; max-width:800px;">
        <div style="background:linear-gradient(135deg,#1a237e 0%,#283593 100%); color:white; 
                    padding:20px; border-radius:8px 8px 0 0;">
            <h2 style="margin:0;">JIT Risk Assessment Dashboard</h2>
            <p style="margin:5px 0 0 0; opacity:0.9;">PO: {po_row['PO_ID']} | Generated: {time.strftime('%Y-%m-%d %H:%M')}</p>
        </div>
        
        <div style="border:1px solid #e0e0e0; border-top:none; padding:20px; background:#fafafa;">
            <div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:20px; margin-bottom:20px;">
                <div style="background:white; padding:15px; border-radius:6px; text-align:center; 
                            border-left:4px solid {tier_colors.get(risk_tier, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Predicted Delay</div>
                    <div style="font-size:2em; font-weight:bold;">{predicted_delay} days</div>
                </div>
                <div style="background:white; padding:15px; border-radius:6px; text-align:center;
                            border-left:4px solid {tier_colors.get(risk_tier, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Risk Tier</div>
                    <div style="font-size:2em; font-weight:bold; color:{tier_colors.get(risk_tier, '#757575')};">{risk_tier}</div>
                </div>
                <div style="background:white; padding:15px; border-radius:6px; text-align:center;
                            border-left:4px solid {action_colors.get(action_level, '#757575')};">
                    <div style="font-size:0.9em; color:#666;">Action Level</div>
                    <div style="font-size:1.5em; font-weight:bold; color:{action_colors.get(action_level, '#757575')};">{action_level}</div>
                </div>
            </div>
            
            <div style="background:white; padding:15px; border-radius:6px; margin-bottom:15px;">
                <h4 style="margin-top:0;">Order Details</h4>
                <table style="width:100%;">
                    <tr><td style="width:30%;"><strong>Vendor:</strong></td><td>{po_row['Vendor_ID']} ({po_row['Vendor_Country']})</td></tr>
                    <tr><td><strong>Material:</strong></td><td>{po_row['Material_ID']} - {po_row['Material_Group']}</td></tr>
                    <tr><td><strong>Criticality:</strong></td><td style="color:{'#c62828' if po_row['Criticality_Flag']=='Yes' else '#666'};">
                        {'CRITICAL' if po_row['Criticality_Flag']=='Yes' else 'Standard'}</td></tr>
                    <tr><td><strong>Quantity:</strong></td><td>{int(po_row['Order_Quantity']):,} units</td></tr>
                    <tr><td><strong>Unit Price:</strong></td><td>${po_row['Net_Price']:.2f}</td></tr>
                    <tr><td><strong>Planned Lead Time:</strong></td><td>{int(po_row['Planned_Lead_Time_Days'])} days</td></tr>
                </table>
            </div>
            
            {alt_table}
        </div>
    </div>
    """
    
    display(HTML(dashboard_html))


# Display the dashboard
display_risk_dashboard(prediction_df.iloc[0], predicted_delay, risk_tier, action_level, proposal)

---

## Checkpoint & Exercise

### What You've Learned

1. **Data Preparation**: How to structure procurement data for tabular AI prediction
2. **Model Inference**: How to call SAP-RPT-1 via AI Core for delay prediction
3. **Business Policy**: How to translate predictions into actionable risk tiers
4. **Mitigation Logic**: How to automatically identify alternative suppliers

### Exercise: Explore Different Scenarios

Go back to **Step 1.9** and try these combinations:

| Scenario | Vendor | Quantity | Expected Result |
|----------|--------|----------|----------------|
| Best case | VENDOR_A | 1000 | Green, no action |
| Moderate risk | VENDOR_B | 1500 | Amber, monitoring |
| Q4 crunch | VENDOR_B (month 11) | 2000 | Higher risk due to seasonality |
| Worst case | VENDOR_C | 2600 | Red, mitigation needed |

### Discussion Questions

1. What additional S/4HANA attributes might improve prediction accuracy?
2. Should the mitigation proposal trigger automatic workflow, or require human approval?
3. How would you deploy this as a real-time service vs. batch scoring?

---

## Summary: Part 1 Complete

You've successfully built a predictive JIT risk assessment pipeline using:

| Component | What It Does |
|-----------|-------------|
| **Historical Data** | Provides context for pattern learning |
| **SAP-RPT-1** | Predicts delivery delay based on PO features |
| **Business Policy** | Translates predictions into action levels |
| **Mitigation Engine** | Identifies and evaluates alternative suppliers |

### What's Next: Part 2

In Part 2, we'll evolve this into an **agentic solution** where:
- An AI agent monitors multiple POs continuously
- When risk is detected, the agent reasons about mitigation options
- The agent generates structured proposals with human-in-the-loop approval
- The solution moves from reactive to proactive supply chain management

---

*Continue to Part 2: `part2_jit_agent.ipynb`*

---

## Cleanup

Run this cell at the end of the workshop to clear sensitive data.

In [ ]:
def cleanup_session():
    """Clean up sensitive data at end of workshop."""
    global aicore_client
    
    print("Cleaning up workshop session...")
    
    # Clear AI Core client token
    if aicore_client is not None:
        aicore_client._access_token = None
        aicore_client._token_expires_at = 0.0
        print("   [OK] Cleared cached OAuth token")
    
    # Clear sensitive env vars from memory
    sensitive_vars = ["AICORE_CLIENT_ID", "AICORE_CLIENT_SECRET"]
    for var in sensitive_vars:
        if var in os.environ:
            del os.environ[var]
    print("   [OK] Cleared sensitive environment variables")
    
    print("\nCleanup complete.")

# Uncomment to run cleanup:
# cleanup_session()